In [ ]:
# CS781 Phase 2 - L-NER on top of the Phase-1 Model 3 (MTP decoder)
# part 1: tokenizers with char offsets, BIO tags, windows, decoding, strict metric
import os
import re
import json
import math
import time
import random
import zipfile
from bisect import bisect_left, bisect_right
from collections import Counter

import numpy as np

T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f}m]", *a, flush=True)


# ---------------- BIO tags ----------------
IGNORE = -100


class TagScheme:
    # O = 0, then B-X / I-X for every label. also builds the legal transitions for viterbi
    def __init__(self, labels):
        self.labels = list(labels)
        self.tags = ["O"] + [f"{p}-{l}" for l in self.labels for p in "BI"]
        self.tag2id = {t: i for i, t in enumerate(self.tags)}
        self.n = len(self.tags)
        self.O = 0
        # kind: 0=O 1=B 2=I, lab: label index (-1 for O)
        self.kind = np.zeros(self.n, dtype=np.int64)
        self.lab = -np.ones(self.n, dtype=np.int64)
        for t, i in self.tag2id.items():
            if t == "O":
                continue
            p, l = t.split("-", 1)
            self.kind[i] = 1 if p == "B" else 2
            self.lab[i] = self.labels.index(l)

        NEG = -1e9
        self.trans = np.zeros((self.n, self.n))  # trans[prev, cur]
        self.start = np.zeros(self.n)
        for cur in range(self.n):
            if self.kind[cur] == 2:
                self.start[cur] = NEG  # can't start with I-X
                for prev in range(self.n):
                    # I-X only after B-X / I-X
                    if not (self.kind[prev] != 0 and self.lab[prev] == self.lab[cur]):
                        self.trans[prev, cur] = NEG

    def span_tag(self, label, begin):
        return self.tag2id[("B-" if begin else "I-") + label]


# ---------------- byte-level BPE tokenizer ----------------
# Phase-1 byte-level BPE (HF tokenizers). We split the text ourselves so every punctuation char is its own
# piece, otherwise a token could cross an entity edge ("U.P." + "(")
L = r"[^\W\d_]"  # a letter
PIECES = re.compile("|".join([
    rf"(?<={L})'(?:s|t|re|ve|m|ll|d)(?!{L})",  # contractions, only after a letter
    rf" ?{L}+",  # (space)letters
    r" ?\d+",  # (space)digits
    r" ?(?:[^\s\w]|_)",  # (space)one punctuation char
    r"\s+(?!\S)",
    r"\s+",
]))


class ByteBPEOffsets:
    def __init__(self, tk):
        self.tk = tk
        vocab = tk.get_vocab()
        self.vocab_size = tk.get_vocab_size()
        self.base_vocab_size = self.vocab_size
        self.pad_id = vocab["<|pad|>"]
        self.unk_id = vocab["<|unk|>"]
        self.eos_id = vocab["<|endoftext|>"]
        self.cache = {}

    def ids_of(self, text):
        return list(self.tk.encode(text).ids)

    def _piece(self, piece):
        hit = self.cache.get(piece)
        if hit is None:
            e = self.tk.encode(piece)
            hit = [(int(i), int(a), int(b)) for i, (a, b) in zip(e.ids, e.offsets)]
            for _, a, b in hit:
                assert 0 <= a <= b <= len(piece), f"bad offsets {a},{b} for piece {piece!r}"
            self.cache[piece] = hit
        return hit

    def encode(self, text):
        # returns ids + char offsets. zero-width tokens get valid=False (fed to the model, never labelled)
        ids, starts, ends = [], [], []
        for m in PIECES.finditer(text):
            base = m.start()
            for tid, a, b in self._piece(m.group()):
                s, e = base + a, base + b
                # offsets include the leading space of "Ġword", trim it
                while s < e and text[s].isspace():
                    s += 1
                while e > s and text[e - 1].isspace():
                    e -= 1
                ids.append(tid)
                starts.append(s)
                ends.append(e)
        # keep offsets monotone so bisect works
        starts = np.maximum.accumulate(np.asarray(starts, dtype=np.int64))
        ends = np.maximum(np.maximum.accumulate(np.asarray(ends, dtype=np.int64)), starts)
        return {"ids": np.asarray(ids, dtype=np.int64), "starts": starts, "ends": ends,
                "valid": ends > starts, "case": np.zeros(len(ids), dtype=np.int64)}


# ---------------- legacy word-level BPE tokenizer ----------------
# Phase-1 word-level BPE ('</w>' markers, lower-cased). Its regex swallowed most ASCII punctuation,
# so here every punctuation char is its own token: reuse the Phase-1 embedding if 'c</w>' exists,
# otherwise add a new embedding row. Case is lost in Phase 1, so a small case class is fed as an extra embedding:
# 0 lower, 1 Capitalised, 2 UPPER, 3 digits, 4 punct, 5 mixed
N_CASE = 6
PRETOK = re.compile(
    r"(?<=[a-zA-Z])'(?:[sStTmMdD]|[rR][eE]|[vV][eE]|[lL][lL])(?![a-zA-Z])"
    r"|[a-zA-Z]+|[0-9]+|[^\sA-Za-z0-9]")


def case_class(piece):
    c0 = piece[0]
    if c0.isascii() and c0.isdigit():
        return 3
    if not (c0.isascii() and c0.isalpha()):
        return 0 if c0 == "'" and len(piece) > 1 else 4
    if piece.islower():
        return 0
    if piece.isupper():
        return 1 if len(piece) == 1 else 2
    if piece[0].isupper() and piece[1:].islower():
        return 1
    return 5


class OffsetBPE:
    def __init__(self, vocab_path, merges_path):
        with open(vocab_path, "r", encoding="utf-8") as f:
            self.vocab = json.load(f)
        with open(merges_path, "r", encoding="utf-8") as f:
            merges = json.load(f)
        self.ranks = {(a, b): i for i, (a, b) in enumerate(merges)}
        self.base_vocab_size = max(self.vocab.values()) + 1  # rows that exist in the Phase-1 checkpoint
        self.vocab_size = self.base_vocab_size  # grows when punctuation tokens are added
        self.pad_id = self.vocab["<|pad|>"]
        self.unk_id = self.vocab["<|unk|>"]
        self.eos_id = self.vocab["<|endoftext|>"]
        self.punct_id = {}
        self.punct_unk = self.unk_id
        self.cache = {}

    def build_punct(self, texts, min_count=3):
        # one token per punctuation char seen in the task texts
        cnt = Counter(ch for t in texts for ch in t if not ch.isspace() and not (ch.isascii() and ch.isalnum()))
        reused, new = 0, 0
        for ch, n in sorted(cnt.items(), key=lambda x: -x[1]):
            if n < min_count:
                continue
            tid = self.vocab.get(ch + "</w>")
            if tid is not None:
                reused += 1
            else:
                tid = self.vocab_size
                self.vocab_size += 1
                new += 1
            self.punct_id[ch] = tid
        self.punct_unk = self.vocab_size  # shared id for rare punctuation
        self.vocab_size += 1
        log(f"punctuation: {len(self.punct_id)} chars ({reused} reuse a Phase-1 embedding, {new} new rows, +1 rare row) "
            f"-> vocab {self.vocab_size} (checkpoint has {self.base_vocab_size})")
        return self

    def _word(self, w):
        hit = self.cache.get(w)
        if hit is not None:
            return hit
        syms = list(w) + ["</w>"]
        while len(syms) > 1:
            # apply the lowest-rank merge available
            best, best_rank = None, None
            for i in range(len(syms) - 1):
                r = self.ranks.get((syms[i], syms[i + 1]))
                if r is not None and (best_rank is None or r < best_rank):
                    best, best_rank = (syms[i], syms[i + 1]), r
            if best is None:
                break
            merged, out, i = best[0] + best[1], [], 0
            while i < len(syms):
                if i < len(syms) - 1 and syms[i] == best[0] and syms[i + 1] == best[1]:
                    out.append(merged)
                    i += 2
                else:
                    out.append(syms[i])
                    i += 1
            syms = out
        res = [(self.vocab.get(s, self.unk_id), len(s) - 4 if s.endswith("</w>") else len(s)) for s in syms]
        self.cache[w] = res
        return res

    def encode(self, text):
        # offsets are on the original text, only each piece is lower-cased (length stays the same)
        ids, starts, ends, case = [], [], [], []
        for m in PRETOK.finditer(text):
            piece = m.group()
            cc = case_class(piece)
            c0 = piece[0]
            is_word = (c0.isascii() and c0.isalnum()) or (c0 == "'" and len(piece) > 1)
            if is_word:
                pos = m.start()
                for tid, n in self._word(piece.lower()):
                    ids.append(tid)
                    starts.append(pos)
                    pos += n
                    ends.append(pos)
                    case.append(cc)
                assert pos == m.end(), f"offset drift in piece {piece!r} at {m.start()}"
            else:
                ids.append(self.punct_id.get(piece, self.punct_unk))
                starts.append(m.start())
                ends.append(m.end())
                case.append(cc)
        starts = np.asarray(starts, dtype=np.int64)
        ends = np.asarray(ends, dtype=np.int64)
        return {"ids": np.asarray(ids, dtype=np.int64), "starts": starts, "ends": ends,
                "valid": ends > starts, "case": np.asarray(case, dtype=np.int64)}


def coverage_check(text, enc):
    # every non-space char should be inside a token. returns (multiply covered, uncovered)
    covered = np.zeros(len(text), dtype=np.int8)
    for s, e in zip(enc["starts"], enc["ends"]):
        covered[s:e] += 1
    dup = int((covered > 1).sum())
    miss = sum(1 for i, ch in enumerate(text) if covered[i] == 0 and not ch.isspace())
    return dup, miss


# ---------------- gold spans <-> token tags ----------------
def spans_to_tags(enc, spans, scheme):
    # a token belongs to a span if at least half of its chars are inside it
    starts, ends, valid = enc["starts"], enc["ends"], enc["valid"]
    tags = np.zeros(len(starts), dtype=np.int64)
    tags[~valid] = IGNORE
    lost = 0
    for s, e, lab in sorted(spans):
        lo = bisect_right(ends, s)  # first token ending after s
        hi = bisect_left(starts, e)  # first token starting at or after e
        sel = []
        for i in range(lo, hi):
            if not valid[i]:
                continue
            overlap = min(e, ends[i]) - max(s, starts[i])
            if overlap * 2 >= (ends[i] - starts[i]):
                sel.append(i)
        if not sel:
            lost += 1
            continue
        for k, i in enumerate(sel):
            tags[i] = scheme.span_tag(lab, k == 0)
    return tags, lost


def tags_to_spans(enc, tag_ids, scheme):
    # a dangling I-X starts a new span
    starts, ends, valid = enc["starts"], enc["ends"], enc["valid"]
    spans, cur = [], None  # cur = [first_tok, last_tok, label_idx]
    for i in range(len(tag_ids)):
        if not valid[i]:
            continue
        t = int(tag_ids[i])
        kind, lab = scheme.kind[t], scheme.lab[t]
        if kind == 0:
            if cur:
                spans.append(cur)
                cur = None
        elif kind == 1 or cur is None or cur[2] != lab:
            if cur:
                spans.append(cur)
            cur = [i, i, lab]
        else:
            cur[1] = i
    if cur:
        spans.append(cur)
    return [(int(starts[a]), int(ends[b]), scheme.labels[l]) for a, b, l in spans]


def viterbi(logp, scheme, o_bias=0.0):
    # best legal BIO path over the valid tokens, logp: (N, n_tags)
    N, K = logp.shape
    if N == 0:
        return np.zeros(0, dtype=np.int64)
    lp = logp.astype(np.float64).copy()
    lp[:, scheme.O] += o_bias
    score = lp[0] + scheme.start
    back = np.zeros((N, K), dtype=np.int16)
    for t in range(1, N):
        cand = score[:, None] + scheme.trans  # (prev, cur)
        back[t] = cand.argmax(0)
        score = cand.max(0) + lp[t]
    path = np.zeros(N, dtype=np.int64)
    path[-1] = int(score.argmax())
    for t in range(N - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path


def decode_probs(enc, probs, scheme, strategy="viterbi", o_bias=0.0):
    # probs: (n_tokens, n_tags) for the whole document -> list of (start, end, label)
    vidx = np.nonzero(enc["valid"])[0]
    p = probs[vidx]
    if strategy == "viterbi":
        path = viterbi(np.log(np.clip(p, 1e-9, 1.0)), scheme, o_bias)
    else:  # greedy argmax
        q = p.copy()
        q[:, scheme.O] *= math.exp(o_bias)
        path = q.argmax(1)
    full = np.zeros(len(enc["ids"]), dtype=np.int64)
    full[vidx] = path
    return tags_to_spans(enc, full, scheme)


# ---------------- windows ----------------
def make_windows(n_tokens, body_len, stride, shift=0):
    # body windows [s, e) over token positions. shift moves the grid (random per epoch = free augmentation)
    out, s = [], -shift
    while True:
        a, e = max(s, 0), min(s + body_len, n_tokens)
        if e > a and (not out or out[-1] != (a, e)):
            out.append((a, e))
        if s + body_len >= n_tokens:
            break
        s += stride
    return out


def window_inputs(ids, win, header_tokens, sep_ids):
    # [header][sep][body], no prefix if the window already starts inside the header
    s, e = win
    ids = np.asarray(ids)
    prefix = []
    if header_tokens > 0 and s >= header_tokens:
        prefix = ids[:header_tokens].tolist() + list(sep_ids)
    return prefix + ids[s:e].tolist(), len(prefix)


def centre_weights(n):
    i = np.arange(n)
    return np.minimum(i + 1, n - i).astype(np.float64)


# ---------------- strict metric ----------------
def strict_scores(gold, pred, labels):
    # start, end and label must all match. gold / pred: {doc_id: [(s, e, label), ...]}
    tp, fp, fn = Counter(), Counter(), Counter()
    for d in set(gold) | set(pred):
        g = set((int(a), int(b), l) for a, b, l, *_ in gold.get(d, []))
        p = set((int(a), int(b), l) for a, b, l, *_ in pred.get(d, []))
        for x in g & p:
            tp[x[2]] += 1
        for x in p - g:
            fp[x[2]] += 1
        for x in g - p:
            fn[x[2]] += 1
    per = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = {"P": P, "R": R, "F1": F, "tp": tp[l], "fp": fp[l], "fn": fn[l]}
    used = [l for l in labels if tp[l] + fp[l] + fn[l] > 0] or list(labels)
    macro = {k: float(np.mean([per[l][k] for l in used])) for k in ("P", "R", "F1")}
    TP, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro_p = TP / (TP + FP) if TP + FP else 0.0
    micro_r = TP / (TP + FN) if TP + FN else 0.0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0
    return {"per_label": per, "macro": macro, "micro_f1": micro_f}


def error_breakdown(gold, pred):
    # why do predictions fail: label confusion, boundary error, spurious (FP) or missed (FN)
    c, conf = Counter(), Counter()
    for d in set(gold) | set(pred):
        g = [(int(a), int(b), l) for a, b, l, *_ in gold.get(d, [])]
        p = [(int(a), int(b), l) for a, b, l, *_ in pred.get(d, [])]
        gs, ps = set(g), set(p)
        for (s, e, l) in ps - gs:
            same_span = [x for x in g if x[0] == s and x[1] == e]
            overl_same = [x for x in g if x[2] == l and x[0] < e and s < x[1]]
            overl_any = [x for x in g if x[0] < e and s < x[1]]
            if same_span:
                c["FP_label_confusion"] += 1
                conf[(same_span[0][2], l)] += 1
            elif overl_same:
                c["FP_boundary_error"] += 1
            elif overl_any:
                c["FP_overlaps_other_label"] += 1
            else:
                c["FP_spurious"] += 1
        for (s, e, l) in gs - ps:
            if any(x[2] == l and x[0] < e and s < x[1] for x in p):
                c["FN_boundary_error"] += 1
            elif any(x[0] == s and x[1] == e for x in p):
                c["FN_label_confusion"] += 1
            elif any(x[0] < e and s < x[1] for x in p):
                c["FN_overlapped_by_other_label"] += 1
            else:
                c["FN_missed"] += 1
    return dict(c), conf.most_common(10)


# ---------------- submission ----------------
def validate_predictions(preds, texts, labels):
    ok_labels = set(labels)
    n = 0
    for d, spans in preds.items():
        T = texts[d]
        last_end = -1
        for s, e, l in sorted((int(a), int(b), l) for a, b, l, *_ in spans):
            assert 0 <= s < e <= len(T), (d, s, e, "out of bounds")
            assert l in ok_labels, (d, l, "unknown label")
            assert s >= last_end, (d, s, e, "overlap")
            assert not T[s].isspace() and not T[e - 1].isspace(), (d, s, e, "span touches whitespace")
            last_end = e
            n += 1
    assert set(preds) == set(texts), "missing / extra document ids"
    return n


def write_submission(preds, texts, labels, out_dir, include_text=False):
    os.makedirs(out_dir, exist_ok=True)
    n = validate_predictions(preds, texts, labels)
    out = {}
    for d, spans in preds.items():
        rows = []
        for s, e, l in sorted((int(a), int(b), l) for a, b, l, *_ in spans):
            rows.append([s, e, l, texts[d][s:e]] if include_text else [s, e, l])
        out[d] = rows
    pj = os.path.join(out_dir, "prediction.json")
    with open(pj, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    zp = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zp, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pj, arcname="prediction.json")  # has to sit at the zip root
    log(f"wrote {pj} and {zp}: {len(out)} documents, {n} entities")
    return pj, zp


# ---------------- data loading ----------------
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P2_FILES = {
    "fold1_text": "Phase_2_LNER/Train/fold1-text.json", "fold1_gold": "Phase_2_LNER/Train/fold1-gold.json",
    "fold2_text": "Phase_2_LNER/Train/fold2-text.json", "fold2_gold": "Phase_2_LNER/Train/fold2-gold.json",
    "labels": "Phase_2_LNER/Train/labels.txt", "fold3_text": "Phase_2_LNER/Test/fold3-text.json",
}


def _hf_token():
    # token comes from the environment or the Kaggle secret named HF (never hard-code it)
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF")
    except Exception:
        return None


def locate_phase2_files():
    # 1) env P2_DIR  2) copy under /kaggle/input  3) download from the HF dataset repo
    by_name = {os.path.basename(v): k for k, v in P2_FILES.items()}
    for root in [os.environ.get("P2_DIR"), "/kaggle/input", os.getcwd()]:
        if not root or not os.path.isdir(root):
            continue
        found = {}
        for d, _, files in os.walk(root):
            for f in files:
                if f in by_name:
                    found[by_name[f]] = os.path.join(d, f)
        if len(found) == len(P2_FILES):
            log(f"Phase-2 files found under {root}")
            return found
    from huggingface_hub import hf_hub_download, login
    tok = _hf_token()
    if tok:
        login(token=tok)
    log("downloading Phase-2 files from the Hugging Face dataset repo")
    return {k: hf_hub_download(repo_id=HF_DATASET_REPO, filename=v, repo_type="dataset", token=tok)
            for k, v in P2_FILES.items()}


def load_phase2():
    files = locate_phase2_files()

    def read(p):
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)

    with open(files["labels"], "r", encoding="utf-8") as f:
        labels = [l.strip() for l in f if l.strip()]
    texts, gold = {}, {}
    for k in ("fold1", "fold2"):
        t, g = read(files[k + "_text"]), read(files[k + "_gold"])
        for d, txt in t.items():
            texts[d] = txt
            gold[d] = [(int(a[0]), int(a[1]), str(a[2])) for a in g.get(d, [])]
    test = read(files["fold3_text"])
    return labels, texts, gold, test


# =============================================================================
# part 2: LoRA (Hu et al. 2021) on the frozen Phase-1 Model 3
# (decoder only, RoPE + GQA + SwiGLU, 4-way MTP heads, 50K vocab) + BIO classifier head
# =============================================================================
import copy
import contextlib
from types import SimpleNamespace

import torch
import torch.nn as nn
import torch.nn.functional as F

# ---------------- config ----------------
SEED = 1234
SMOKE = os.environ.get("SMOKE", "0") == "1"  # tiny end-to-end run to check the pipeline
EXP_NAME = os.environ.get("EXP_NAME", "exp4_model3_lora")
OUT_DIR = f"/kaggle/working/phase2_lner_{EXP_NAME}" if os.path.isdir("/kaggle/working") else f"./phase2_lner_{EXP_NAME}"
MODEL_TAG = "decoder_only_rope_gqa_swiglu_mtp4_v1"  # tag written by the Phase-1 Model-3 notebook
TOKENIZER_KIND = os.environ.get("TOKENIZER_KIND", "auto")  # auto | legacy | bytebpe
ALLOW_RANDOM_INIT = os.environ.get("ALLOW_RANDOM_INIT", "0") == "1"  # ablation only

# data / windows
VAL_DOCS = 10  # docs held out of the 70 for model selection
HEADER_TOKENS = 128  # start of the case (parties, case no.) is repeated in front of every later window
BODY_LEN = 512
TRAIN_STRIDE = 320
INF_STRIDE = 256

# model
BIDIRECTIONAL = os.environ.get("BIDIRECTIONAL", "1") == "1"  # Phase-1 model is causal, drop the mask for NER
FT_DROPOUT = 0.1
HEAD_DROPOUT = 0.1

# LoRA: y = W0 x + (alpha / r) * B(A(x)), B starts at zero so step 0 equals the Phase-1 model
LORA_R = int(os.environ.get("LORA_R", 16))
LORA_ALPHA = float(os.environ.get("LORA_ALPHA", 32))
LORA_DROPOUT = float(os.environ.get("LORA_DROPOUT", 0.05))
LORA_TARGETS = tuple(os.environ.get("LORA_TARGETS", "q_proj,k_proj,v_proj,o_proj,w_gate,w_up,w_down").split(","))
MERGE_FOR_INFERENCE = True  # fold B@A into W before test inference and check nothing changed

# optimisation (only LoRA + classifier are trained)
EPOCHS = 20
PATIENCE = 5
BATCH = 8
LR_LORA = float(os.environ.get("LR_LORA", 5e-4))
LR_HEAD = 1e-3
WEIGHT_DECAY = 0.01  # classifier weight only
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
CLASS_WEIGHT_POWER = 0.25  # w = (count_O / count_tag) ** power, capped. 0 = plain CE
CLASS_WEIGHT_CAP = 6.0
REFIT_ON_ALL = False  # retrain on all 70 docs for the best epoch count
INCLUDE_TEXT_IN_PREDICTION = False
AMP_ENABLED = os.environ.get("NO_AMP", "0") != "1"  # NO_AMP=1 -> fp32 (if the loss goes NaN on fp16 GPUs)


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


# ---------------- Phase-1 architecture ----------------
# same module / parameter names as the Phase-1 notebook so the checkpoint loads cleanly
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        var = x.float().pow(2).mean(-1, keepdim=True)
        inv_rms = torch.rsqrt(var + self.eps).to(x.dtype)
        return x * inv_rms * self.weight.to(x.dtype)


def build_rope(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(max_len).float(), inv_freq)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    cos, sin = cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


def repeat_kv(x, groups):
    if groups == 1:
        return x
    B, KV, T, hd = x.shape
    return x[:, :, None].expand(B, KV, groups, T, hd).reshape(B, KV * groups, T, hd)


class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.h, self.kv = cfg.n_heads, cfg.n_kv_heads
        self.hd = cfg.d_model // cfg.n_heads
        self.groups = self.h // self.kv
        self.q_proj = nn.Linear(cfg.d_model, self.h * self.hd, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv * self.hd, bias=False)
        self.o_proj = nn.Linear(self.h * self.hd, cfg.d_model, bias=False)

    def forward(self, x, cos, sin, attn_mask, causal):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.h, self.hd).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        k, v = repeat_kv(k, self.groups), repeat_kv(v, self.groups)
        if causal:  # padding is on the right, so no mask needed
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        else:  # bidirectional, only hide the padding keys
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask)
        return self.o_proj(out.transpose(1, 2).reshape(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)

    def forward(self, x):
        return self.w_down(F.silu(self.w_gate(x)) * self.w_up(x))


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.attn = RMSNorm(cfg.d_model), GQAttention(cfg)
        self.norm2, self.ffn = RMSNorm(cfg.d_model), SwiGLU(cfg)
        self.drop = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin, attn_mask, causal):
        x = x + self.drop(self.attn(self.norm1(x), cos, sin, attn_mask, causal))
        return x + self.drop(self.ffn(self.norm2(x)))


class MTPHead(nn.Module):
    # Model 3 has no final norm, head 0 (the normal next-token head) plays that role here
    def __init__(self, dim):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.norm = RMSNorm(dim)

    def forward(self, x):
        return self.norm(x + self.proj(x))


class Backbone(nn.Module):
    # Phase-1 DecoderLM: embed, blocks, heads.0 (the other MTP heads are dropped).
    # legacy tokenizer only: new_embed (extra punctuation rows) and case_embed (zero init)
    def __init__(self, cfg, n_new_tokens=0, use_case=False):
        super().__init__()
        self.max_len = cfg.max_len
        self.base_vocab = cfg.real_vocab  # ids >= real_vocab are padding rows
        self.embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.new_embed = nn.Embedding(n_new_tokens, cfg.d_model) if n_new_tokens > 0 else None
        self.case_embed = nn.Embedding(N_CASE, cfg.d_model) if use_case else None
        if self.case_embed is not None:
            nn.init.zeros_(self.case_embed.weight)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.heads = nn.ModuleList([MTPHead(cfg.d_model)])
        self.drop = nn.Dropout(cfg.dropout)
        cos, sin = build_rope(cfg.d_model // cfg.n_heads, cfg.max_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, idx, case, key_mask, causal):
        B, T = idx.shape
        assert T <= self.max_len, f"sequence {T} > Phase-1 max_len {self.max_len}"
        cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
        x = self.embed(idx.clamp(max=self.base_vocab - 1))
        if self.new_embed is not None:
            is_new = (idx >= self.base_vocab).unsqueeze(-1)
            x = torch.where(is_new, self.new_embed((idx - self.base_vocab).clamp(min=0)), x)
        if self.case_embed is not None:
            x = x + self.case_embed(case)
        x = self.drop(x)
        attn_mask = None if causal else key_mask[:, None, None, :]
        for blk in self.blocks:
            x = blk(x, cos, sin, attn_mask, causal)
        return self.heads[0](x)


class LNERModel(nn.Module):
    def __init__(self, cfg, n_tags, causal, n_new_tokens=0, use_case=False):
        super().__init__()
        self.backbone = Backbone(cfg, n_new_tokens, use_case)
        self.head_drop = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(cfg.d_model, n_tags)
        self.causal = causal
        nn.init.normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

    def forward(self, ids, case, mask):
        return self.classifier(self.head_drop(self.backbone(ids, case, mask, self.causal)))


def phase1_state(ckpt):
    # drop MTP heads 1.. and strip torch.compile / DataParallel prefixes
    sd = {k.replace("_orig_mod.", "").replace("module.", ""): v for k, v in ckpt["model"].items()}
    return {k: v for k, v in sd.items() if not (k.startswith("heads.") and not k.startswith("heads.0."))}


# ---------------- LoRA ----------------
class LoRALinear(nn.Module):
    # frozen linear + trainable low-rank update
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        assert base.bias is None, "Phase-1 projections have no bias"
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        self.scale = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    def forward(self, x):
        y = self.base(x)
        if self.merged:
            return y
        return y + F.linear(F.linear(self.drop(x), self.lora_A), self.lora_B) * self.scale

    @torch.no_grad()
    def merge(self):
        # W <- W0 + (alpha / r) B A, so inference costs the same as the plain layer
        if not self.merged:
            self.base.weight.add_(self.lora_B @ self.lora_A, alpha=self.scale)
            self.merged = True


def apply_lora(model):
    # freeze the backbone (except new embeddings), wrap the target layers, keep the classifier trainable
    for name, p in model.backbone.named_parameters():
        p.requires_grad = name.startswith(("new_embed", "case_embed"))
    wrapped = 0
    for blk in model.backbone.blocks:
        for parent in (blk.attn, blk.ffn):
            for name in LORA_TARGETS:
                layer = getattr(parent, name, None)
                if isinstance(layer, nn.Linear):
                    setattr(parent, name, LoRALinear(layer, LORA_R, LORA_ALPHA, LORA_DROPOUT))
                    wrapped += 1
    assert wrapped > 0, f"LORA_TARGETS {LORA_TARGETS} matched no layer"
    for p in model.classifier.parameters():
        p.requires_grad = True
    return wrapped


def merge_lora(model):
    for m in model.modules():
        if isinstance(m, LoRALinear):
            m.merge()


def trainable_snapshot(model):
    # only what training changed (a few MB instead of the whole model)
    return {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    lora = sum(p.numel() for n, p in model.named_parameters() if "lora_" in n)
    return total, train, lora


# ---------------- finding Phase-1 files ----------------
def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def find_phase1_ckpt():
    # best.pt / latest.pt written by the Model-3 notebook (tag must match so Model 1/2 are never picked)
    env = os.environ.get("PHASE1_CKPT")
    if env and os.path.exists(env):
        return env
    cands = []
    for root in ["/kaggle/input", "/kaggle/working", os.getcwd()]:
        if not os.path.isdir(root):
            continue
        for d, _, files in os.walk(root):
            for name, meta_name in (("best.pt", "best_meta.json"), ("latest.pt", "latest_meta.json")):
                if name not in files:
                    continue
                meta = read_json(os.path.join(d, meta_name)) or {}
                if meta.get("tag") != MODEL_TAG:
                    continue
                rank = (0 if name == "best.pt" else 1, meta.get("val_loss", meta.get("best_val", 1e9)))
                cands.append((rank, os.path.join(d, name)))
    return min(cands)[1] if cands else None


def import_tokenizer():
    try:
        from tokenizers import Tokenizer
    except ImportError:
        os.system("pip -q install tokenizers")
        from tokenizers import Tokenizer
    return Tokenizer


def find_tokenizers(real_vocab, all_texts):
    # every Phase-1 compatible tokenizer under /kaggle/input. Both kinds have a 50K vocab,
    # so the size alone can't say which one the checkpoint was trained with
    cands = {}
    roots = [r for r in ["/kaggle/input", "/kaggle/working", os.getcwd()] if os.path.isdir(r)]
    env = os.environ.get("TOKENIZER_JSON")
    tj = [env] if env and os.path.exists(env) else []
    legacy_dirs = []
    for root in roots:
        for d, _, files in os.walk(root):
            if "tokenizer.json" in files:
                tj.append(os.path.join(d, "tokenizer.json"))
            for v, m in (("vocab_50k.json", "merges_50k.json"), ("vocab.json", "merges.json")):
                if v in files and m in files:
                    legacy_dirs.append((os.path.join(d, v), os.path.join(d, m)))

    if TOKENIZER_KIND in ("auto", "bytebpe"):
        Tokenizer = import_tokenizer()
        for p in tj:
            try:
                tk = Tokenizer.from_file(p)
            except Exception:
                continue
            if tk.get_vocab_size() == real_vocab:
                cands["bytebpe"] = (ByteBPEOffsets(tk), p)
                break
    if TOKENIZER_KIND in ("auto", "legacy"):
        for vp, mp in legacy_dirs:
            v = read_json(vp)
            if v and max(v.values()) + 1 == real_vocab:
                t = OffsetBPE(vp, mp)
                t.build_punct(all_texts)  # punctuation was invisible in Phase 1 -> new embedding rows
                cands["legacy"] = (t, vp)
                break
    if not cands:
        raise FileNotFoundError(f"no tokenizer with {real_vocab} entries found (kind={TOKENIZER_KIND}). Attach tokenizer.json "
                                "(byte-level) or vocab_50k.json + merges_50k.json (word-level), or set TOKENIZER_JSON")
    return cands


@torch.no_grad()
def lm_probe_loss(ckpt, cfg, tok, texts, device, n_docs=8, n_tok=384, skip=16):
    # next-token loss of the pretrained model on Phase-2 text. With the right tokenizer it is close to the
    # checkpoint's own val_loss, with the wrong one it is near ln(vocab) ~ 10.8
    bb = Backbone(cfg, 0, False)
    bb.load_state_dict(phase1_state(ckpt), strict=True)
    bb = bb.to(device).eval()
    base, losses = cfg.real_vocab, []
    for t in texts[:n_docs]:
        piece = t[len(t) // 3: len(t) // 3 + 8000]
        ids = tok.encode(piece)["ids"]
        ids = ids[ids < base][:n_tok + 1]  # drop the new punctuation ids (legacy), Phase 1 never saw them
        if len(ids) < skip + 8:
            continue
        x = torch.tensor(ids[:-1], dtype=torch.long, device=device)[None]
        y = torch.tensor(ids[1:], dtype=torch.long, device=device)
        z = bb(x, torch.zeros_like(x), None, True)
        logits = F.linear(z, bb.embed.weight[:base])[0].float()
        losses.append(F.cross_entropy(logits, y, reduction="none")[skip:].mean().item())
    del bb
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return float(np.mean(losses)) if losses else float("nan")


def choose_tokenizer(ckpt, cfg, texts, test_texts, device):
    cands = find_tokenizers(cfg.real_vocab, list(texts.values()) + list(test_texts.values()))
    ref = ckpt.get("val_loss") if ckpt else None
    probe = {}
    for kind, (t, path) in cands.items():
        probe[kind] = lm_probe_loss(ckpt, cfg, t, list(texts.values()), device) if ckpt else float("nan")
        log(f"tokenizer probe: {kind:8s} ({path}) -> next-token loss {probe[kind]:.3f} (checkpoint val_loss = {ref})")
    kind = min(probe, key=lambda k: probe[k]) if ckpt else next(iter(cands))
    if ckpt and (math.isnan(probe[kind]) or probe[kind] > 7.0 or (ref is not None and probe[kind] > ref + 3.5)):
        msg = (f"no tokenizer reproduces the checkpoint's LM loss (best: {kind} {probe[kind]:.2f} vs val_loss {ref}). "
               "The checkpoint was probably trained with another tokenizer. Attach the exact files used by the Model-3 "
               "notebook (vocab_50k.json + merges_50k.json) and/or set TOKENIZER_KIND. FORCE_TOKENIZER=1 turns this into a warning")
        if os.environ.get("FORCE_TOKENIZER", "0") != "1":
            raise RuntimeError(msg)
        log("WARNING: " + msg)
    if len(cands) > 1:
        other = [k for k in cands if k != kind][0]
        log(f"-> using '{kind}' (loss {probe[kind]:.2f}) instead of '{other}' (loss {probe[other]:.2f})")
    else:
        log(f"-> only '{kind}' available (loss {probe[kind]:.2f})")
    return kind, cands[kind][0], cands[kind][1], probe


# ---------------- batching ----------------
def build_windows_for_docs(doc_ids, ctx, shift, stride):
    items = []
    zeros = [0] * len(ctx.sep_ids)
    for d in doc_ids:
        enc, tags = ctx.enc[d], ctx.tags[d]
        for (s, e) in make_windows(len(enc["ids"]), BODY_LEN, stride, shift):
            ids, off = window_inputs(enc["ids"], (s, e), HEADER_TOKENS, ctx.sep_ids)
            cs, _ = window_inputs(enc["case"], (s, e), HEADER_TOKENS, zeros)
            lab = [IGNORE] * off + [int(x) for x in tags[s:e]]
            items.append((ids, cs, lab))
    return items


def collate(items, pad_id, device):
    T = max(len(x[0]) for x in items)
    B = len(items)
    ids = torch.full((B, T), pad_id, dtype=torch.long)
    case = torch.zeros((B, T), dtype=torch.long)
    lab = torch.full((B, T), IGNORE, dtype=torch.long)
    mask = torch.zeros((B, T), dtype=torch.bool)
    for i, (a, c, l) in enumerate(items):
        n = len(a)
        ids[i, :n] = torch.tensor(a)
        case[i, :n] = torch.tensor(c)
        lab[i, :n] = torch.tensor(l)
        mask[i, :n] = True
    return ids.to(device), case.to(device), lab.to(device), mask.to(device)


# ---------------- inference on whole documents ----------------
@torch.no_grad()
def infer_docs(model, doc_ids, enc_map, ctx, bs=24):
    # overlapping windows, probabilities averaged with centre weights
    model.eval()
    out = {}
    for d in doc_ids:
        enc = enc_map[d]
        n = len(enc["ids"])
        acc = np.zeros((n, ctx.scheme.n), dtype=np.float32)
        wsum = np.zeros(n, dtype=np.float32)
        wins = make_windows(n, BODY_LEN, INF_STRIDE)
        for i in range(0, len(wins), bs):
            chunk = wins[i:i + bs]
            items, offs = [], []
            for (s, e) in chunk:
                ids, off = window_inputs(enc["ids"], (s, e), HEADER_TOKENS, ctx.sep_ids)
                cs, _ = window_inputs(enc["case"], (s, e), HEADER_TOKENS, [0] * len(ctx.sep_ids))
                items.append((ids, cs, [IGNORE] * len(ids)))
                offs.append(off)
            ids_t, case_t, _, mask_t = collate(items, ctx.pad_id, ctx.device)
            with ctx.autocast():
                logits = model(ids_t, case_t, mask_t)
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
            for b, (s, e) in enumerate(chunk):
                w = centre_weights(e - s).astype(np.float32)
                acc[s:e] += probs[b, offs[b]:offs[b] + (e - s)] * w[:, None]
                wsum[s:e] += w
        out[d] = acc / np.maximum(wsum, 1e-9)[:, None]
    return out


def decode_docs(doc_ids, probs, enc_map, scheme, strategy, o_bias):
    return {d: decode_probs(enc_map[d], probs[d], scheme, strategy, o_bias) for d in doc_ids}


# ---------------- training helpers ----------------
def class_weights(train_ids, ctx):
    cnt = np.zeros(ctx.scheme.n)
    for d in train_ids:
        t = ctx.tags[d]
        cnt += np.bincount(t[t != IGNORE], minlength=ctx.scheme.n)
    w = np.ones(ctx.scheme.n)
    if CLASS_WEIGHT_POWER > 0:
        for i in range(1, ctx.scheme.n):
            if cnt[i] > 0:
                w[i] = min(CLASS_WEIGHT_CAP, (cnt[0] / cnt[i]) ** CLASS_WEIGHT_POWER)
    log(f"tag counts: O={int(cnt[0])} entity tokens={int(cnt[1:].sum())} ({100 * cnt[1:].sum() / cnt.sum():.1f}%)")
    log("class weights: O=1.00 " + " ".join(f"{ctx.scheme.tags[i]}={w[i]:.2f}" for i in range(1, ctx.scheme.n)))
    return torch.tensor(w, dtype=torch.float32, device=ctx.device)


def make_param_groups(model):
    named = [(n, p) for n, p in model.named_parameters() if p.requires_grad]
    lora = [p for n, p in named if "lora_" in n]
    newp = [p for n, p in named if n.startswith(("backbone.new_embed", "backbone.case_embed"))]
    head_w = [p for n, p in named if n == "classifier.weight"]
    head_b = [p for n, p in named if n == "classifier.bias"]
    assert lora and head_w and head_b
    allowed = lambda n: "lora_" in n or n.startswith(("classifier", "backbone.new_embed", "backbone.case_embed"))
    assert all(not p.requires_grad for n, p in model.named_parameters() if not allowed(n)), "a Phase-1 weight is trainable"
    groups = [{"params": lora, "lr": LR_LORA, "base_lr": LR_LORA, "weight_decay": 0.0},
              {"params": head_w, "lr": LR_HEAD, "base_lr": LR_HEAD, "weight_decay": WEIGHT_DECAY},
              {"params": head_b, "lr": LR_HEAD, "base_lr": LR_HEAD, "weight_decay": 0.0}]
    if newp:
        groups.append({"params": newp, "lr": LR_HEAD, "base_lr": LR_HEAD, "weight_decay": 0.0})
    return groups


def lr_mult(step, total, warm):
    # linear warmup, then cosine down to 5%
    if step < warm:
        return (step + 1) / warm
    prog = (step - warm) / max(1, total - warm)
    return 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * prog))


def init_model(ctx, ckpt):
    cfg = copy.deepcopy(ctx.cfg)
    cfg.dropout = FT_DROPOUT
    model = LNERModel(cfg, ctx.scheme.n, causal=not BIDIRECTIONAL, n_new_tokens=ctx.n_new, use_case=ctx.use_case)
    if ckpt is not None:
        missing, unexpected = model.backbone.load_state_dict(phase1_state(ckpt), strict=False)
        assert not unexpected, f"unexpected keys in the Phase-1 checkpoint: {unexpected}"
        assert set(missing) <= {"new_embed.weight", "case_embed.weight"}, f"Phase-1 weights missing: {missing}"
        if model.backbone.new_embed is not None:  # new punctuation rows: mean Phase-1 embedding + noise
            with torch.no_grad():
                mu = model.backbone.embed.weight[:cfg.real_vocab].mean(0, keepdim=True)
                model.backbone.new_embed.weight.copy_(mu + 0.02 * torch.randn_like(model.backbone.new_embed.weight))
        log(f"loaded Phase-1 Model-3 weights ({cfg.n_layers} blocks, MTP head 0); new rows: {ctx.n_new}; case feature: {ctx.use_case}")
    else:
        log("WARNING: training from RANDOM INIT (ALLOW_RANDOM_INIT), not valid for the assignment")
    n_wrapped = apply_lora(model)  # after loading, wrapping renames q_proj.weight -> q_proj.base.weight
    log(f"LoRA: r={LORA_R} alpha={LORA_ALPHA} dropout={LORA_DROPOUT} on {n_wrapped} layers ({','.join(LORA_TARGETS)})")
    return model.to(ctx.device)


def merge_and_check(model, d0, ctx, strategy="viterbi", o_bias=0.0):
    # fold LoRA into the base weights and make sure the outputs stay the same
    t = time.time()
    before = infer_docs(model, [d0], ctx.enc, ctx)[d0]
    unmerged_sec = time.time() - t
    merge_lora(model)
    t = time.time()
    after = infer_docs(model, [d0], ctx.enc, ctx)[d0]
    merged_sec = time.time() - t
    same = (decode_probs(ctx.enc[d0], before, ctx.scheme, strategy, o_bias)
            == decode_probs(ctx.enc[d0], after, ctx.scheme, strategy, o_bias))
    report = {"max_abs_prob_diff": float(np.abs(before - after).max()), "same_spans": same,
              "doc_seconds_unmerged": unmerged_sec, "doc_seconds_merged": merged_sec}
    log(f"merge check on doc {d0}: max |dp| = {report['max_abs_prob_diff']:.2e}, identical spans: {same}, "
        f"inference {unmerged_sec:.1f}s -> {merged_sec:.1f}s")
    return report


# ---------------- diagnostics ----------------
def dataset_diagnostics(ctx, texts, gold):
    ids = list(ctx.enc)
    nt = np.array([len(ctx.enc[d]["ids"]) for d in ids])
    nc = np.array([len(texts[d]) for d in ids])
    zero = int(sum((~ctx.enc[d]["valid"]).sum() for d in ids))
    log(f"tokens/doc: mean {nt.mean():.0f} median {np.median(nt):.0f} p95 {np.percentile(nt, 95):.0f} max {nt.max()} | "
        f"chars/token {nc.sum() / nt.sum():.2f} | zero-width tokens {100 * zero / nt.sum():.1f}%")
    wins = sum(len(make_windows(n, BODY_LEN, TRAIN_STRIDE)) for n in nt)
    log(f"all labelled docs: {wins} windows of <= {HEADER_TOKENS + len(ctx.sep_ids) + BODY_LEN} tokens")
    dup = miss = 0
    for d in ids:
        a, b = coverage_check(texts[d], ctx.enc[d])
        dup += a
        miss += b
    log(f"char coverage: {miss} uncovered chars (must be 0), {dup} multiply covered chars")

    # oracle: gold -> tags -> spans should give back the gold (upper bound of the whole pipeline)
    oracle, n_gold, n_exact, examples = {}, 0, 0, []
    for d in ids:
        t = ctx.tags[d]
        probs = np.zeros((len(t), ctx.scheme.n), dtype=np.float32)
        probs[np.arange(len(t)), np.where(t == IGNORE, 0, t)] = 1.0
        oracle[d] = decode_probs(ctx.enc[d], probs, ctx.scheme, "viterbi")
        g, o = set(gold[d]), set(oracle[d])
        n_gold += len(g)
        n_exact += len(g & o)
        for s, e, l in sorted(g - o):
            if len(examples) < 8:
                near = [x for x in oracle[d] if x[0] < e and s < x[1]]
                context = texts[d][max(0, s - 6):s] + "[" + texts[d][s:e] + "]" + texts[d][e:e + 6]
                examples.append((l, context, [texts[d][x[0]:x[1]] for x in near]))
    sc = strict_scores({d: gold[d] for d in oracle}, oracle, ctx.scheme.labels)
    log(f"ORACLE round trip: strict macro-F1 {sc['macro']['F1']:.4f}, micro-F1 {sc['micro_f1']:.4f}, "
        f"{n_gold - n_exact}/{n_gold} gold spans not reproduced (edge falls inside a token)")
    for l, context, near in examples:
        log("   gold", l, "in context", repr(context), "-> oracle gives", near)
    return sc["micro_f1"]


# ---------------- main pieces shared by both versions ----------------
def setup_device():
    use_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if use_cuda else "cpu")
    amp_dtype, use_scaler = None, False
    if use_cuda and AMP_ENABLED:
        if torch.cuda.get_device_capability()[0] >= 8:
            amp_dtype = torch.bfloat16
        else:
            amp_dtype, use_scaler = torch.float16, True  # older GPUs need loss scaling
    if amp_dtype:
        autocast = lambda: torch.autocast("cuda", dtype=amp_dtype)
    else:
        autocast = lambda: contextlib.nullcontext()
    return device, amp_dtype, use_scaler, autocast


def load_data():
    labels, texts, gold, test_texts = load_phase2()
    if SMOKE:
        keep = sorted(texts)[:8]
        texts = {d: texts[d][:8000] for d in keep}
        gold = {d: [g for g in gold[d] if g[1] <= 8000] for d in keep}
        test_texts = {d: test_texts[d][:8000] for d in sorted(test_texts)[:2]}
    log(f"{len(texts)} labelled docs, {len(test_texts)} test docs, {sum(len(g) for g in gold.values())} gold entities, labels={labels}")
    return labels, texts, gold, test_texts


def load_phase1():
    ck_path = find_phase1_ckpt()
    if ck_path is None and not ALLOW_RANDOM_INIT:
        raise FileNotFoundError(f"Phase-1 Model-3 checkpoint (best.pt + best_meta.json with tag '{MODEL_TAG}') not found. "
                                "Attach the notebook output that has cs781_phase1_model3/checkpoints, or set PHASE1_CKPT=/path/to/best.pt")
    ckpt = torch.load(ck_path, map_location="cpu", weights_only=False) if ck_path else None
    if ckpt is not None:
        log(f"Phase-1 checkpoint: {ck_path} (val_loss={ckpt.get('val_loss')}, epoch={ckpt.get('epoch')}, step={ckpt.get('global_step')})")
        base_rows = phase1_state(ckpt)["embed.weight"].shape[0]
        saved_cfg = ckpt.get("config") or {}
    else:
        base_rows, saved_cfg = 50048, {}
    defaults = dict(vocab_size=base_rows, real_vocab=50000, d_model=640, n_layers=8, n_heads=10, n_kv_heads=2,
                    ffn_hidden=1536, dropout=0.1, max_len=1088, pad_id=0, unk_id=1, eos_id=2)
    defaults.update(saved_cfg)
    cfg = SimpleNamespace(**defaults)
    log(f"Phase-1 config: d_model={cfg.d_model} layers={cfg.n_layers} heads={cfg.n_heads}/{cfg.n_kv_heads} ffn={cfg.ffn_hidden} "
        f"embedding rows={cfg.vocab_size} real vocab={cfg.real_vocab} mtp_heads={getattr(cfg, 'mtp_heads', '?')}")
    return ck_path, ckpt, cfg


def build_ctx(ckpt, cfg, scheme, texts, test_texts, gold, device, autocast, use_scaler):
    # pick the tokenizer, encode every doc once (ids + char offsets) and run the sanity checks
    kind, tok, tok_path, probe = choose_tokenizer(ckpt, cfg, texts, test_texts, device)
    use_case = kind == "legacy"
    n_new = tok.vocab_size - tok.base_vocab_size if use_case else 0
    sep_ids = tok.ids_of("\n") if kind == "bytebpe" else [tok.eos_id]  # a token the model saw in Phase 1
    assert HEADER_TOKENS + len(sep_ids) + BODY_LEN <= cfg.max_len
    log(f"tokenizer kind={kind}: vocab={tok.vocab_size} pad={tok.pad_id} eos={tok.eos_id} separator={sep_ids} "
        f"new rows={n_new} case feature={use_case}")

    ctx = SimpleNamespace(scheme=scheme, tok=tok, cfg=cfg, device=device, pad_id=tok.pad_id, sep_ids=sep_ids,
                          autocast=autocast, use_scaler=use_scaler, enc={}, tags={}, lost={}, gold=gold,
                          n_new=n_new, use_case=use_case, kind=kind, tok_path=tok_path, probe=probe)
    for d, t in texts.items():
        ctx.enc[d] = tok.encode(t)
        ctx.tags[d], ctx.lost[d] = spans_to_tags(ctx.enc[d], gold[d], scheme)
    test_enc = {d: tok.encode(t) for d, t in test_texts.items()}
    unc = {d: coverage_check(t, test_enc[d])[1] for d, t in test_texts.items()}
    if any(unc.values()):  # harmless (spans come from token edges) but good to know
        log(f"note: {sum(unc.values())} test chars not covered by any token: {[d for d, v in unc.items() if v][:5]}")
    ctx.oracle_micro = dataset_diagnostics(ctx, texts, gold)
    if ctx.oracle_micro < 0.99:
        log("WARNING: oracle round trip below 0.99, check the spans above before trusting any score")
    return ctx, test_enc


def choose_decoding(val_ids, probs, ctx):
    # try greedy and viterbi with a few O biases, keep the best macro-F1 on the validation docs
    labels = ctx.scheme.labels
    vgold = {d: ctx.gold[d] for d in val_ids}
    results = []
    for strat, ob in [("greedy", 0.0)] + [("viterbi", b) for b in (-1.0, -0.5, 0.0, 0.5, 1.0, 1.5)]:
        pred = decode_docs(val_ids, probs, ctx.enc, ctx.scheme, strat, ob)
        sc = strict_scores(vgold, pred, labels)
        results.append((sc["macro"]["F1"], strat, ob, sc, pred))
        log(f"decode {strat:8s} o_bias={ob:+.1f}: macro-F1 {sc['macro']['F1']:.4f}  P {sc['macro']['P']:.4f}  "
            f"R {sc['macro']['R']:.4f}  micro-F1 {sc['micro_f1']:.4f}")
    results.sort(key=lambda r: -r[0])
    _, strategy, o_bias, sc, pred = results[0]
    log(f"chosen decoding: {strategy}, o_bias={o_bias:+.1f} (on {len(val_ids)} validation docs)")
    log("per-label validation (strict):")
    for l in labels:
        r = sc["per_label"][l]
        log(f"   {l:10s} P {r['P']:.3f} R {r['R']:.3f} F1 {r['F1']:.3f}   tp {r['tp']:4d} fp {r['fp']:4d} fn {r['fn']:4d}")
    errors, conf = error_breakdown(vgold, pred)
    log(f"error taxonomy on validation: {errors}")
    log(f"top label confusions (gold, predicted): {conf}")
    report = {"scores": sc, "errors": errors, "confusions": [[list(k), v] for k, v in conf],
              "decode": {"strategy": strategy, "o_bias": o_bias}, "val_docs": val_ids,
              "val_tokens": int(sum(len(ctx.enc[d]["ids"]) for d in val_ids))}
    return strategy, o_bias, report


def config_snapshot():
    return {name: v for name, v in globals().items() if name.isupper() and isinstance(v, (int, float, str, bool))}


# ---------------- training ----------------
def run_training(model, train_ids, val_ids, epochs, ctx, tag):
    rng = random.Random(SEED)
    cw = class_weights(train_ids, ctx)
    opt = torch.optim.AdamW(make_param_groups(model), betas=(0.9, 0.98), eps=1e-8)
    scaler = make_scaler(ctx.use_scaler)
    steps_per_epoch = math.ceil(len(build_windows_for_docs(train_ids, ctx, 0, TRAIN_STRIDE)) / BATCH)
    total = steps_per_epoch * epochs
    warm = max(1, int(WARMUP_FRAC * total))
    log(f"[{tag}] {len(train_ids)} train docs, ~{steps_per_epoch * BATCH} windows/epoch, "
        f"{steps_per_epoch} steps/epoch, {total} total steps")

    best = {"f1": -1, "epoch": 0, "state": None}
    bad, step = 0, 0
    history = []
    for ep in range(1, epochs + 1):
        model.train()
        # random grid shift every epoch = free augmentation
        items = build_windows_for_docs(train_ids, ctx, rng.randrange(TRAIN_STRIDE), TRAIN_STRIDE)
        rng.shuffle(items)
        t_ep, run_loss, nb = time.time(), 0.0, 0
        for i in range(0, len(items), BATCH):
            ids, case, lab, mask = collate(items[i:i + BATCH], ctx.pad_id, ctx.device)
            m = lr_mult(step, total, warm)
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * m
            with ctx.autocast():
                logits = model(ids, case, mask)
            loss = F.cross_entropy(logits.float().view(-1, ctx.scheme.n), lab.view(-1), weight=cw, ignore_index=IGNORE)
            if not torch.isfinite(loss):
                log(f"[{tag}] non-finite loss at step {step}, batch skipped (if it repeats, rerun with NO_AMP=1)")
                opt.zero_grad(set_to_none=True)
                step += 1
                continue
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item()
            nb += 1
            step += 1

        row = {"epoch": ep, "train_loss": run_loss / max(1, nb), "sec": time.time() - t_ep}
        if val_ids:
            probs = infer_docs(model, val_ids, ctx.enc, ctx)
            pred = decode_docs(val_ids, probs, ctx.enc, ctx.scheme, "viterbi", 0.0)
            sc = strict_scores({d: ctx.gold[d] for d in val_ids}, pred, ctx.scheme.labels)
            row.update({"val_macro_f1": sc["macro"]["F1"], "val_macro_p": sc["macro"]["P"],
                        "val_macro_r": sc["macro"]["R"], "val_micro_f1": sc["micro_f1"]})
            if sc["macro"]["F1"] > best["f1"]:
                best.update(f1=sc["macro"]["F1"], epoch=ep, state=trainable_snapshot(model))
                bad = 0
                torch.save(best["state"], os.path.join(OUT_DIR, f"adapter_{tag}.pt"))
            else:
                bad += 1
        history.append(row)
        log(f"[{tag}] epoch {ep}/{epochs} "
            + " ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}" for k, v in row.items() if k != "epoch")
            + ("  <- best" if val_ids and best["epoch"] == ep else ""))
        if val_ids and bad >= PATIENCE:
            log(f"[{tag}] early stop (no val improvement for {PATIENCE} epochs)")
            break

    if val_ids and best["state"] is not None:
        model.load_state_dict(best["state"], strict=False)  # only trainable tensors are stored
    else:
        best["epoch"] = epochs
    return best, history


# ---------------- main ----------------
def main():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    os.makedirs(OUT_DIR, exist_ok=True)

    device, amp_dtype, use_scaler, autocast = setup_device()
    use_cuda = device.type == "cuda"
    log(f"device={device} amp={amp_dtype} smoke={SMOKE} exp={EXP_NAME}")

    labels, texts, gold, test_texts = load_data()
    scheme = TagScheme(labels)
    ck_path, ckpt, cfg = load_phase1()
    ctx, test_enc = build_ctx(ckpt, cfg, scheme, texts, test_texts, gold, device, autocast, use_scaler)

    # split (same seed -> same validation docs in every run)
    all_ids = sorted(texts)
    val_ids = sorted(random.Random(SEED).sample(all_ids, VAL_DOCS)) if not SMOKE else all_ids[-2:]
    train_ids = [d for d in all_ids if d not in val_ids]
    log(f"split: {len(train_ids)} train docs / {len(val_ids)} validation docs {val_ids}")

    # model + training
    model = init_model(ctx, ckpt)
    n_total, n_train, n_lora = count_params(model)
    log(f"parameters: total {n_total / 1e6:.2f}M | trainable {n_train / 1e6:.3f}M = {100 * n_train / n_total:.2f}% "
        f"(LoRA {n_lora / 1e6:.3f}M + classifier {(n_train - n_lora) / 1e3:.1f}K) | frozen Phase-1 {(n_total - n_train) / 1e6:.2f}M")
    if use_cuda:
        torch.cuda.reset_peak_memory_stats()
    t_train = time.time()
    epochs = 1 if SMOKE else EPOCHS
    best, history = run_training(model, train_ids, val_ids, epochs, ctx, EXP_NAME)
    train_sec = time.time() - t_train
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if use_cuda else 0.0
    log(f"training done in {train_sec / 60:.1f} min, best epoch {best['epoch']} "
        f"(val strict macro-F1 {best['f1']:.4f}), peak GPU memory {peak_gb:.2f} GB")

    # pick the decoding on the validation docs
    t_inf = time.time()
    vprobs = infer_docs(model, val_ids, ctx.enc, ctx)
    inf_sec = time.time() - t_inf
    strategy, o_bias, val_report = choose_decoding(val_ids, vprobs, ctx)
    val_report["inference_sec_val"] = inf_sec

    adapter_mb = os.path.getsize(os.path.join(OUT_DIR, f"adapter_{EXP_NAME}.pt")) / 1e6
    log(f"adapter checkpoint (LoRA + classifier): {adapter_mb:.1f} MB vs full model ~{n_total * 4 / 1e6:.0f} MB in fp32")

    # optional refit on every labelled doc
    if REFIT_ON_ALL and not SMOKE:
        log(f"refitting from the Phase-1 weights on all {len(all_ids)} docs for {best['epoch']} epochs")
        model = init_model(ctx, ckpt)
        run_training(model, all_ids, [], best["epoch"], ctx, EXP_NAME + "_all")

    # merge adapters into the base weights and check nothing changed
    merge_report = {}
    if MERGE_FOR_INFERENCE:
        merge_report = merge_and_check(model, val_ids[0], ctx, strategy, o_bias)

    # test predictions -> submission
    t_inf = time.time()
    tprobs = infer_docs(model, list(test_texts), test_enc, ctx)
    preds = decode_docs(list(test_texts), tprobs, test_enc, scheme, strategy, o_bias)
    test_sec = time.time() - t_inf
    cnt = Counter(l for v in preds.values() for _, _, l in v)
    log(f"test: {sum(len(v) for v in preds.values())} entities in {len(preds)} docs ({test_sec:.0f}s); per label {dict(cnt)}")
    pj, zp = write_submission(preds, test_texts, labels, OUT_DIR, include_text=INCLUDE_TEXT_IN_PREDICTION)

    summary = {"experiment": EXP_NAME, "phase1_ckpt": ck_path, "tokenizer": ctx.tok_path, "tokenizer_kind": ctx.kind,
               "tokenizer_probe_loss": ctx.probe, "phase1_val_loss": (ckpt or {}).get("val_loss"), "phase1_tag": MODEL_TAG,
               "config": config_snapshot(), "method": "lora",
               "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "targets": LORA_TARGETS},
               "params_total": n_total, "params_trainable": n_train, "params_lora": n_lora, "adapter_mb": adapter_mb,
               "merge_check": merge_report, "peak_gpu_gb": peak_gb, "train_minutes": train_sec / 60,
               "test_inference_sec": test_sec, "best_epoch": best["epoch"], "history": history,
               "validation": val_report, "oracle_micro_f1": ctx.oracle_micro, "vocab": ctx.tok.vocab_size}
    with open(os.path.join(OUT_DIR, "run_summary.json"), "w") as f:
        json.dump(summary, f, indent=1, default=str)
    log(f"done. Submit: {zp}")


if __name__ == "__main__":
    main()
